In [1]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

# Build a tiny BPE model and train it
tok = Tokenizer(models.BPE())
tok.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.BpeTrainer(show_progress=False, min_frequency=2)

corpus = [
    "roses are red",
    "are we sure are we",
    "BERT is big",
    "GPT-2 is so",
    "so is it",
]

tok.train_from_iterator(corpus, trainer=trainer)

print("Vocab size:", tok.get_vocab_size())
print("Tokens for 'are is red':", tok.encode("are is red").tokens)

Vocab size: 25
Tokens for 'are is red': ['are', 'is', 're', 'd']


In [2]:
from pathlib import Path
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

# Small demo corpus – duplicate to make merges actually happen
corpus = [
    "Roses are red, violets are blue. BPE loves ByteLevel, and so do you!",
    "Tokenizers are fun. Training BPE with ByteLevel is robust on any text.",
    "Exact log-likelihood scoring may pick different merges than count.",
    "Approximate ΔLL is often a fast heuristic that tracks the exact score.",
    "Emoji 😀 and accented words like café should work with ByteLevel.",
    "Lowercasing is not applied here; we are testing the raw byte stream.",
] * 200

SPECIAL_TOKENS = ["<pad>", "<s>", "</s>", "<unk>"]


In [3]:
def train_bpe_variant(
    name: str,
    score_by: str,
    stop_by: str,
    vocab_size: int = 1200,
    min_frequency: int = 2,
    add_prefix_space: bool = True,
    sample_texts: list[str] | None = None,
) -> Tokenizer:
    print(f"\n=== Training {name} ===")
    print(
        f"  score_by={score_by}, stop_by={stop_by}, "
        f"vocab_size={vocab_size}, min_frequency={min_frequency}, "
        f"ByteLevel(add_prefix_space={add_prefix_space})"
    )

    # 1) Empty BPE model + ByteLevel pre-tokenizer & decoder
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=add_prefix_space)
    tok.decoder = decoders.ByteLevel()

    # 2) Trainer (IMPORTANT: don't pass kwargs as None—just omit them)
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=min_frequency,
        show_progress=True,
        special_tokens=SPECIAL_TOKENS,
        score_by=score_by,            # "count" | "exact_ll" | "approx_ll"
        stop_by=stop_by,              # "vocab_size" | "delta_ll_exact" | "delta_ll_approx"
        # You can also pass: limit_alphabet=..., max_token_length=..., initial_alphabet=[...]
        # continuing_subword_prefix="##", end_of_word_suffix="</w>"  # only if you want them
        # track_ll=True,              
    )

    # 3) Train
    tok.train_from_iterator(corpus, trainer=trainer)

    # 4) Quick encode tests
    tests = sample_texts or [
        "Roses are red and so are bytes.",
        "Tokenizers 🧪 test with ByteLevel!",
        "I had coffee at the café.",
        "This exact-approx demo compares merge choices.",
    ]
    for t in tests:
        enc = tok.encode(t)
        print(f"\n[{name}] Text:   {t!r}")
        print(f"[{name}] Tokens: {enc.tokens}")
        print(f"[{name}] IDs:    {enc.ids}")

    # 5) Save model files and show first 20 merges
    out_dir = Path(f"out_{name}")
    out_dir.mkdir(parents=True, exist_ok=True)
    saved = tok.model.save(str(out_dir), name)  # returns [vocab.json, merges.txt]
    print(f"\n[{name}] Saved files:")
    for p in saved:
        print("   ", p)

    merges_file = next((Path(p) for p in saved if str(p).endswith(".txt")), None)
    if merges_file and merges_file.exists():
        print(f"\n[{name}] First 20 merges:")
        with merges_file.open("r", encoding="utf-8") as fh:
            for i, line in enumerate(fh):
                if i == 0 and line.startswith("#version"):
                    continue  # skip header
                if i > 20:
                    break
                print(f"  {i:>3}: {line.strip()}")

    return tok


In [5]:
# Classic BPE (frequency) – baseline
tok_count = train_bpe_variant(
    name="bpe_count",
    score_by="count",
    stop_by="vocab_size",
    vocab_size=1200,
    min_frequency=2,
    add_prefix_space=True,
)


=== Training bpe_count ===
  score_by=count, stop_by=vocab_size, vocab_size=1200, min_frequency=2, ByteLevel(add_prefix_space=True)




[bpe_count] Text:   'Roses are red and so are bytes.'
[bpe_count] Tokens: ['ĠRoses', 'Ġare', 'Ġred', 'Ġand', 'Ġso', 'Ġare', 'Ġbyte', 's', '.']
[bpe_count] IDs:    [243, 62, 233, 108, 192, 62, 206, 33, 7]

[bpe_count] Text:   'Tokenizers 🧪 test with ByteLevel!'
[bpe_count] Tokens: ['ĠTokenizers', 'ĠðŁ', 'Ġte', 'st', 'Ġwith', 'ĠByteLevel', '!']
[bpe_count] IDs:    [255, 182, 99, 55, 102, 76, 4]

[bpe_count] Text:   'I had coffee at the café.'
[bpe_count] Tokens: ['Ġ', 'Ġh', 'a', 'd', 'Ġco', 'ff', 'e', 'e', 'Ġa', 't', 'Ġthe', 'ĠcafÃ©', '.']
[bpe_count] IDs:    [45, 97, 16, 19, 165, 123, 20, 20, 50, 34, 106, 236, 7]

[bpe_count] Text:   'This exact-approx demo compares merge choices.'
[bpe_count] Tokens: ['ĠT', 'h', 'is', 'Ġexact', '-', 'a', 'pp', 'ro', 'x', 'Ġd', 'e', 'mo', 'Ġco', 'm', 'p', 'a', 're', 's', 'Ġmer', 'g', 'e', 'Ġ', 'c', 'ho', 'ic', 'es', '.

In [6]:
# Greedy exact ΔLL – select by exact gain; early stop when best ΔLL <= 0
tok_exact = train_bpe_variant(
    name="bpe_exact_ll",
    score_by="exact_ll",
    stop_by="delta_ll_exact",
    vocab_size=1200,    # still acts as a hard cap
    min_frequency=2,
    add_prefix_space=True,
)


=== Training bpe_exact_ll ===
  score_by=exact_ll, stop_by=delta_ll_exact, vocab_size=1200, min_frequency=2, ByteLevel(add_prefix_space=True)




[bpe_exact_ll] Text:   'Roses are red and so are bytes.'
[bpe_exact_ll] Tokens: ['ĠRoses', 'Ġare', 'Ġred', 'Ġand', 'Ġso', 'Ġare', 'Ġbyte', 's', '.']
[bpe_exact_ll] IDs:    [220, 72, 260, 118, 291, 72, 224, 33, 7]

[bpe_exact_ll] Text:   'Tokenizers 🧪 test with ByteLevel!'
[bpe_exact_ll] Tokens: ['ĠTokenizers', 'Ġ', 'ðŁ', 'Ġ', 'test', 'Ġwith', 'ĠByteLevel', '!']
[bpe_exact_ll] IDs:    [137, 45, 59, 45, 162, 123, 73, 4]

[bpe_exact_ll] Text:   'I had coffee at the café.'
[bpe_exact_ll] Tokens: ['Ġ', 'Ġ', 'h', 'a', 'd', 'Ġ', 'co', 'ff', 'e', 'e', 'Ġa', 't', 'Ġthe', 'ĠcafÃ©', '.']
[bpe_exact_ll] IDs:    [45, 45, 23, 16, 19, 45, 172, 237, 20, 20, 282, 34, 149, 213, 7]

[bpe_exact_ll] Text:   'This exact-approx demo compares merge choices.'
[bpe_exact_ll] Tokens: ['ĠT', 'h', 'is', 'Ġexact', '-', 'a', 'pp', 'r', 'o', 'x', 'Ġ', 'd', 'e', 'm', 'o', '

In [7]:
# Greedy approx ΔLL – select by approximation; early stop by approx-ΔLL
tok_approx = train_bpe_variant(
    name="bpe_approx_ll",
    score_by="approx_ll",
    stop_by="delta_ll_approx",
    vocab_size=1200,
    min_frequency=2,
    add_prefix_space=True,
)


=== Training bpe_approx_ll ===
  score_by=approx_ll, stop_by=delta_ll_approx, vocab_size=1200, min_frequency=2, ByteLevel(add_prefix_space=True)




[bpe_approx_ll] Text:   'Roses are red and so are bytes.'
[bpe_approx_ll] Tokens: ['ĠRoses', 'Ġare', 'Ġred', 'Ġand', 'Ġso', 'Ġare', 'Ġbyte', 's', '.']
[bpe_approx_ll] IDs:    [256, 61, 269, 71, 273, 61, 244, 33, 7]

[bpe_approx_ll] Text:   'Tokenizers 🧪 test with ByteLevel!'
[bpe_approx_ll] Tokens: ['ĠTokenizers', 'Ġ', 'ðŁ', 'Ġ', 'te', 'st', 'Ġwith', 'ĠByteLevel', '!']
[bpe_approx_ll] IDs:    [202, 45, 75, 45, 55, 72, 97, 83, 4]

[bpe_approx_ll] Text:   'I had coffee at the café.'
[bpe_approx_ll] Tokens: ['Ġ', 'Ġ', 'h', 'a', 'd', 'Ġ', 'c', 'o', 'f', 'f', 'e', 'e', 'Ġa', 't', 'Ġthe', 'ĠcafÃ©', '.']
[bpe_approx_ll] IDs:    [45, 45, 23, 16, 19, 45, 18, 30, 21, 21, 20, 20, 60, 34, 89, 242, 7]

[bpe_approx_ll] Text:   'This exact-approx demo compares merge choices.'
[bpe_approx_ll] Tokens: ['ĠT', 'h', 'is', 'Ġexact', '-', 'a', 'pp', 'r', 'o', 